In [1]:
# CELL 1 - install (run once per fresh runtime)
# Pillow is intentionally NOT in this list. Colab already ships a working Pillow; upgrading it
# is what triggered the uninstall -> restart -> reload-secrets loop.
# If pip output below says it replaced Pillow, do Runtime > Restart session once, then continue from cell 2.
!pip -q install -U "transformers>=4.49.0" accelerate fastapi uvicorn python-multipart pyngrok nest-asyncio pillow-heif

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 72.5 MB/s eta 0:00:00


In [2]:
# CELL 2 - imports + config + secrets  (run this first after EVERY restart)
import os, io, gc, re, sys, json, time, logging, threading, traceback
import numpy as np
import requests
import torch
import PIL
from PIL import Image, ImageOps

# Extra formats: HEIC/HEIF (iPhone photos) and AVIF, if pillow-heif is installed
HEIF_OK = False
try:
    import pillow_heif
    pillow_heif.register_heif_opener()
    if hasattr(pillow_heif, "register_avif_opener"):
        pillow_heif.register_avif_opener()
    HEIF_OK = True
except Exception as e:
    print("pillow-heif not available (HEIC/AVIF uploads will be rejected):", e)

Image.init()  # make sure every built-in format plugin is registered

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
PORT = 8000
MAX_UPLOAD_BYTES = 20 * 1024 * 1024
MAX_SIDE = 2048          # longest image side sent to the model (keeps GPU memory sane)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def _secret(name):
    value = os.environ.get(name, "")
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name) or ""
        except Exception:
            value = ""
    if value:
        os.environ[name] = value
    return value

HF_TOKEN = _secret("HF_TOKEN")
NGROK_AUTH_TOKEN = _secret("NGROK_AUTH_TOKEN")

# Real-time logs: everything goes to the notebook output immediately
logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True,
                    format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("vlm")

print("Pillow:", PIL.__version__, "| HEIC/AVIF:", HEIF_OK)
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
print("HF token loaded:", bool(HF_TOKEN), "| ngrok token loaded:", bool(NGROK_AUTH_TOKEN))

Pillow: 11.3.0 | HEIC/AVIF: True
Device: cuda | Tesla T4
HF token loaded: True | ngrok token loaded: True


In [3]:
# CELL 3 - load model (slow, do it once)
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

print(f"Loading {MODEL_NAME} on {DEVICE} ...")
processor = AutoProcessor.from_pretrained(MODEL_NAME, token=HF_TOKEN or None)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN or None,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE == "cpu":
    model = model.to(DEVICE)
model.eval()
print("Model loaded.")

23:46:27 | INFO | NumExpr defaulting to 2 threads.
Loading Qwen/Qwen2.5-VL-3B-Instruct on cuda ...
23:46:28 | INFO | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
23:46:28 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
23:46:28 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
23:46:28 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-3B-Instruct/66285546d2b821cf421d4f5eb2576359d3770cd3/preprocessor_config.json "HTTP/1.1 200 OK"
23:46:28 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-3B-Instruct/66285546d2b821cf421d4f5eb2576359d3770cd3/preprocessor_config.json "HTTP/1.1 200 OK"


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

23:46:29 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
23:46:29 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
23:46:29 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-3B-Instruct/66285546d2b821cf421d4f5eb2576359d3770cd3/preprocessor_config.json "HTTP/1.1 200 OK"
23:46:29 | INFO | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-VL-3B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
23:46:29 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
23:46:29 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/chat_template.json "HTTP/1.1 307 Temporary Redirect"
23:46

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

23:46:29 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
23:46:29 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/audio_tokenizer_config.json "HTTP/1.1 404 Not Found"
23:46:29 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
23:46:29 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
23:46:29 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-3B-Instruct/66285546d2b821cf421d4f5eb2576359d3770cd3/preprocessor_config.json "HTTP/1.1 200 OK"
23:46:30 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
23:46:30 | INFO | HTTP Request: HEAD https://hu

...

23:46:34 | INFO | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-VL-3B-Instruct/xet-read-token/66285546d2b821cf421d4f5eb2576359d3770cd3 "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

23:48:45 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
23:48:45 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-3B-Instruct/66285546d2b821cf421d4f5eb2576359d3770cd3/generation_config.json "HTTP/1.1 200 OK"
23:48:45 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-3B-Instruct/66285546d2b821cf421d4f5eb2576359d3770cd3/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Model loaded.


In [4]:
# CELL 4 - inference
def run_qwen_vl(image: Image.Image, query: str, max_new_tokens: int = 256) -> str:
    messages = [{
        "role": "user",
        "content": [{"type": "image", "image": image}, {"type": "text", "text": query.strip()}],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt")
    inputs = {k: v.to(DEVICE) if hasattr(v, "to") else v for k, v in inputs.items()}

    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    trimmed = [out[len(inp):] for inp, out in zip(inputs["input_ids"], generated)]
    return processor.batch_decode(trimmed, skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)[0].strip()

In [5]:
# CELL 5 - accept ANY image: decide by the file's bytes, not by extension / Content-Type
class ImageLoadError(ValueError):
    pass


def load_any_image(data: bytes, max_side: int = MAX_SIDE):
    """bytes -> (RGB PIL image, info dict). Handles JPEG/PNG/WEBP/GIF/BMP/TIFF/ICO/... and HEIC/AVIF."""
    if not data:
        raise ImageLoadError("Uploaded file is empty (0 bytes). In Postman set the 'image' row type to File and pick a file.")

    try:
        img = Image.open(io.BytesIO(data))      # format is sniffed from the bytes
        fmt, orig_size, orig_mode = img.format, img.size, img.mode
        frames = getattr(img, "n_frames", 1)     # GIF / multi-page TIFF: first frame is used
        img.load()                               # full decode (catches truncated files)
    except Exception as e:
        head = data[:12]
        hint = ""
        if head.startswith(b"%PDF"):
            hint = " This is a PDF, not an image."
        elif head[:1] in (b"{", b"<"):
            hint = " This looks like JSON/HTML text, not image bytes."
        raise ImageLoadError(f"Could not decode image ({type(e).__name__}: {e}).{hint} "
                             f"Received {len(data)} bytes, starting with {head!r}.")

    try:
        img = ImageOps.exif_transpose(img)       # fix sideways phone photos
    except Exception:
        pass

    # 16-bit / float images -> 8-bit grayscale
    if img.mode in ("I", "I;16", "I;16L", "I;16B", "I;16N", "F"):
        arr = np.asarray(img).astype("float32")
        if img.mode == "F":
            arr = (arr - arr.min()) / ((arr.max() - arr.min()) or 1.0) * 255
        elif arr.max() > 255:
            arr = arr / 257.0
        img = Image.fromarray(np.clip(arr, 0, 255).astype("uint8"))

    # Transparent PNG/WEBP/GIF: flatten on WHITE (plain convert("RGB") gives a black background)
    if img.mode in ("RGBA", "LA", "PA") or (img.mode == "P" and "transparency" in img.info):
        rgba = img.convert("RGBA")
        flat = Image.new("RGB", rgba.size, (255, 255, 255))
        flat.paste(rgba, mask=rgba.getchannel("A"))
        img = flat
    else:
        img = img.convert("RGB")                 # also covers CMYK, L, 1, P

    w, h = img.size
    if min(w, h) < 56:                            # Qwen needs >= 28 px per side after patching
        s = 56 / min(w, h)
        img = img.resize((max(56, round(w * s)), max(56, round(h * s))), Image.Resampling.LANCZOS)
    if max(img.size) > max_side:
        img.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)

    info = {"format": fmt, "mode": orig_mode, "frames": frames,
            "original_size": list(orig_size), "sent_to_model": list(img.size)}
    return img, info

In [6]:
# CELL 6 - FastAPI app
from fastapi import FastAPI, File, Form, HTTPException, Request, UploadFile
from fastapi.responses import JSONResponse
from starlette.concurrency import run_in_threadpool

app = FastAPI(title="Qwen-VL Multimodal API", version="2.0.0",
              description="Image + natural-language query API running on Google Colab.")
infer_lock = threading.Lock()   # one GPU inference at a time

EXPENSE_PROMPT = """
You are processing a bill, receipt, invoice, or expense image.

The user's original query is:
{query}

Answer the original query, AND extract expenses visible in the image.

Use exactly these 5 top-level expense categories:
1. food - meals, team lunches, client dinners, restaurant purchases, groceries and food items (rice, dal, oil, milk, eggs, ...)
2. travel - flights, taxis, mileage, parking, transportation
3. medical - doctor visits, prescriptions, medicines, medical services
4. accommodation - hotels, lodging, extended stays
5. other - office supplies and miscellaneous expenses that do not fit the first four

Return ONLY valid JSON. Do not use Markdown fences and do not add any text before or after the JSON.

Required JSON shape:
{{
  "answer": "direct answer to the original user query",
  "expenses": {{
    "food": "",
    "travel": "",
    "medical": "",
    "accommodation": "",
    "other": {{}}
  }}
}}

Rules for expenses:
- Copy amounts with the currency symbol exactly as printed on the bill (e.g. Rs, INR, or the rupee sign). Never change the currency.
- If a category has no matching expense, use an empty string "".
- If a category has exactly one clear expense, its value may be a string such as "50".
- If a category has multiple items, use an object mapping the item name to its amount, e.g. "food": {{"team lunch": "50", "coffee": "8"}}
- "other" must be an object when it has expenses.
- Do not invent expenses, names, or amounts.
- Keep category names exactly: food, travel, medical, accommodation, other.
""".strip()


def _parse_expense_json(raw: str, original_query: str) -> dict:
    text = (raw or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text).strip()

    parsed = None
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        s, e = text.find("{"), text.rfind("}")
        if s >= 0 and e > s:
            try:
                parsed = json.loads(text[s:e + 1])
            except json.JSONDecodeError:
                parsed = None

    cats = {"food": "", "travel": "", "medical": "", "accommodation": "", "other": {}}
    if isinstance(parsed, dict):
        expenses = parsed.get("expenses", {})
        if isinstance(expenses, dict):
            for c in cats:
                if c in expenses:
                    v = expenses[c]
                    if c == "other":
                        cats[c] = v if isinstance(v, dict) else ({} if v in ("", None) else {"item": str(v)})
                    else:
                        cats[c] = v if isinstance(v, (str, dict)) else str(v)
        return {"answer": str(parsed.get("answer", "")).strip() or original_query, "expenses": cats}
    return {"answer": (raw or "").strip(), "expenses": cats}


def run_expense_query(image: Image.Image, query: str, max_new_tokens: int = 512) -> dict:
    log.info("waiting for GPU slot ...")
    with infer_lock:
        log.info("generating (max_new_tokens=%d) ...", max_new_tokens)
        t = time.perf_counter()
        raw = run_qwen_vl(image, EXPENSE_PROMPT.format(query=query.strip()), max_new_tokens)
        log.info("generation done in %.2fs", time.perf_counter() - t)
    log.info("raw model output: %s", raw.replace("\n", " ")[:800])
    return _parse_expense_json(raw, query.strip())


@app.middleware("http")
async def log_requests(request: Request, call_next):
    t = time.perf_counter()
    who = request.headers.get("x-forwarded-for") or (request.client.host if request.client else "?")
    log.info(">> %s %s  from %s", request.method, request.url.path, who)
    try:
        response = await call_next(request)
    except Exception:
        log.error("unhandled error\n%s", traceback.format_exc())
        raise
    log.info("<< %s %s -> %s  (%.2fs)", request.method, request.url.path, response.status_code, time.perf_counter() - t)
    return response


@app.get("/")
async def root():
    return {"service": "Qwen-VL Multimodal API", "status": "running", "endpoint": "POST /ask"}


@app.get("/health")
async def health():
    return {
        "status": "ok",
        "model_loaded": "model" in globals(),
        "model": MODEL_NAME,
        "device": DEVICE,
        "pillow": PIL.__version__,
        "heic_avif": HEIF_OK,
        "readable_formats": sorted(Image.OPEN.keys()),
    }


@app.post("/ask")
async def ask(query: str = Form(...),
              image: UploadFile | None = File(None),
              file: UploadFile | None = File(None)):     # 'file' accepted too, in case the Postman key is named that
    upload = image if image is not None else file
    if upload is None:
        raise HTTPException(422, "Send the picture as form-data key 'image' with type File.")
    if not query.strip():
        raise HTTPException(422, "The 'query' field is required.")

    data = await upload.read()
    log.info("upload: name=%r content_type=%r bytes=%d first_bytes=%r",
             upload.filename, upload.content_type, len(data), data[:8])
    if len(data) > MAX_UPLOAD_BYTES:
        raise HTTPException(413, f"Image too large (max {MAX_UPLOAD_BYTES // (1024 * 1024)} MB).")

    try:
        pil_image, info = load_any_image(data)
    except ImageLoadError as e:
        log.warning("decode failed: %s", e)
        raise HTTPException(400, str(e))
    log.info("decoded: %s", info)

    start = time.perf_counter()
    try:
        result = await run_in_threadpool(run_expense_query, pil_image, query, 512)
        return JSONResponse({
            "success": True,
            "query": query,
            "answer": result["answer"],
            "expenses": result["expenses"],
            # "image": info,
            # "model": MODEL_NAME,
            # "device": DEVICE,
            # "inference_time_seconds": round(time.perf_counter() - start, 3),
        })
    except torch.cuda.OutOfMemoryError:
        log.error("CUDA OOM")
        raise HTTPException(500, "GPU out of memory. Try a smaller image.")
    except Exception as e:
        log.error("inference error\n%s", traceback.format_exc())
        raise HTTPException(500, f"Inference failed: {type(e).__name__}: {e}")
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [7]:
# CELL 7 - start the server (safe to re-run: stops the old one first)
import nest_asyncio, uvicorn
nest_asyncio.apply()

if "server" in globals() and getattr(server, "started", False):
    server.should_exit = True
    time.sleep(2)

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning", access_log=False))
threading.Thread(target=server.run, daemon=True).start()

for _ in range(40):
    if server.started:
        break
    time.sleep(0.5)
print("Local health:", requests.get(f"http://127.0.0.1:{PORT}/health", timeout=10).json())

23:48:47 | INFO | >> GET /health  from 127.0.0.1
23:48:47 | INFO | << GET /health -> 200  (0.00s)
Local health: {'status': 'ok', 'model_loaded': True, 'model': 'Qwen/Qwen2.5-VL-3B-Instruct', 'device': 'cuda', 'pillow': '11.3.0', 'heic_avif': True, 'readable_formats': ['AVIF', 'BLP', 'BMP', 'BUFR', 'CUR', 'DCX', 'DDS', 'DIB', 'EPS', 'FITS', 'FLI', 'FTEX', 'GBR', 'GIF', 'GRIB', 'HDF5', 'HEIF', 'ICNS', 'ICO', 'IM', 'IMT', 'IPTC', 'JPEG', 'JPEG2000', 'MCIDAS', 'MPEG', 'MSP', 'PCD', 'PCX', 'PIXAR', 'PNG', 'PPM', 'PSD', 'QOI', 'SGI', 'SPIDER', 'SUN', 'TGA', 'TIFF', 'WEBP', 'WMF', 'XBM', 'XPM', 'XVTHUMB']}


In [8]:
# CELL 8 - public URL via ngrok
from pyngrok import ngrok

if not NGROK_AUTH_TOKEN:
    raise RuntimeError("NGROK_AUTH_TOKEN missing - add it in Colab Secrets (Notebook access ON) and re-run cell 2.")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()                                    # drop stale tunnels
public_url = ngrok.connect(PORT, "http").public_url

r = requests.get(public_url + "/health", headers={"ngrok-skip-browser-warning": "1"}, timeout=20)
print("=" * 60)
print("Public URL :", public_url)
print("POST here  :", public_url + "/ask")
print("Docs       :", public_url + "/docs")
print("Health via tunnel ->", r.status_code)
print("=" * 60)
print("Requests now show up live in the output of cell 7 / below.")

23:48:48 | INFO | Updating authtoken for default "config_path" of "ngrok_path": /root/.config/ngrok/ngrok
23:48:48 | INFO | Opening tunnel named: http-8000-ac52c5d3-bdf3-49dd-b04d-0b0cc8362f98
23:48:48 | INFO | t=2026-09-24T23:48:48+0000 lvl=info msg="no configuration paths supplied"
23:48:48 | INFO | t=2026-09-24T23:48:48+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
23:48:48 | INFO | t=2026-09-24T23:48:48+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=<nil>
23:48:48 | INFO | t=2026-09-24T23:48:48+0000 lvl=info msg="FIPS 140 mode" enabled=false
23:48:48 | INFO | t=2026-09-24T23:48:48+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
23:48:48 | INFO | t=2026-09-24T23:48:48+0000 lvl=info msg="client session established" obj=tunnels.session
23:48:48 | INFO | t=2026-09-24T23:48:48+0000 lvl=info msg="tunnel session started" obj=tunnels.session
23:48:48 | INFO | t=2026-09-24T23

In [9]:
from google.colab import drive
drive.mount('/content/drive')

23:49:01 | INFO | >> GET /docs  from 2409:40f4:450d:82d7:258a:8d44:97d:9335
23:49:01 | INFO | << GET /docs -> 200  (0.00s)
23:49:01 | INFO | t=2026-09-24T23:49:01+0000 lvl=info msg="join connections" obj=join id=01db55d7a647 l=127.0.0.1:8000 r=[2409:40f4:450d:82d7:258a:8d44:97d:9335]:53949
23:49:01 | INFO | >> GET /openapi.json  from 2409:40f4:450d:82d7:258a:8d44:97d:9335
23:49:01 | INFO | << GET /openapi.json -> 200  (0.01s)
23:49:23 | INFO | t=2026-09-24T23:49:23+0000 lvl=info msg="join connections" obj=join id=0d9d08c837b1 l=127.0.0.1:8000 r=[2409:40f4:450d:82d7:258a:8d44:97d:9335]:63960
23:49:23 | INFO | >> POST /ask  from 2409:40f4:450d:82d7:258a:8d44:97d:9335
23:49:23 | INFO | upload: name='food_grocery_bill.jpg' content_type='image/jpeg' bytes=45777 first_bytes=b'\xff\xd8\xff\xe0\x00\x10JF'
23:49:23 | INFO | decoded: {'format': 'JPEG', 'mode': 'RGB', 'frames': 1, 'original_size': [265, 425], 'sent_to_model': [265, 425]}
23:49:23 | INFO | waiting for GPU slot ...
23:49:23 | INFO 

ValueError: mount failed

In [ ]:
# CELL 9 (optional) - self-test: every format must decode; then one real call
# Upload food_grocery_bill.jpg to Colab (Files panel) first.
TEST_IMAGE = "/content/food_grocery_bill.jpg"

src = Image.open(TEST_IMAGE).convert("RGB")
tests = [("JPEG", "jpg"), ("PNG", "png"), ("WEBP", "webp"), ("BMP", "bmp"), ("GIF", "gif"), ("TIFF", "tiff")]
if HEIF_OK:
    tests.append(("HEIF", "heic"))

for fmt, ext in tests:
    try:
        buf = io.BytesIO(); src.save(buf, fmt)
        _, info = load_any_image(buf.getvalue())
        print(f"OK   {fmt:5s} ->", info)
    except Exception as e:
        print(f"FAIL {fmt:5s} ->", type(e).__name__, e)

buf = io.BytesIO(); src.convert("RGBA").save(buf, "PNG")     # transparent-style PNG
print("OK   RGBA PNG ->", load_any_image(buf.getvalue())[1])

with open(TEST_IMAGE, "rb") as f:
    r = requests.post(f"http://127.0.0.1:{PORT}/ask",
                      files={"image": ("food_grocery_bill.jpg", f, "image/jpeg")},
                      data={"query": "What is this bill? Give the total."})
print(r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

## Postman
`POST https://<ngrok-url>/ask` -> Body -> **form-data**

| Key | Type | Value |
|---|---|---|
| `image` | **File** | any jpg / png / webp / bmp / gif / tiff / heic |
| `query` | Text | `What is this bill?` |

Do not set `Content-Type` manually.

## Troubleshooting
- **404 through ngrok**: watch the notebook output. If no `>> POST /ask` line appears, the request never reached FastAPI (wrong/stale URL, or the tunnel points at a different port than cell 7). Re-run cell 8 and use the newly printed URL. If the line appears with `-> 404`, the path is wrong (must be `/ask`).
- **400 "Could not decode image"**: the message now shows the real decode error and the first bytes of what was received.
- **Pillow problems**: don't upgrade/uninstall Pillow. If pip replaced it in cell 1, restart the session once and continue from cell 2.